# Margin Engine — Planner Distillation

Fine-tunes a small open model to translate an engineer's question into an **Analysis Plan**
for the Industrial Copilot.

**What is being taught, and what is not.** The model learns `intent -> plan`. It is never
taught facts about machines, and it never emits a number. Baking data into weights is the
exact failure this architecture exists to eliminate: weights cannot be updated, audited, or
unit-checked. The plan is validated against a semantic layer before it executes, and every
figure in the final answer is computed and verified downstream. So a weak model here
produces a *worse plan*, never a wrong number.

**Why the reward is free.** Training pairs come from plans that actually validated and
executed. There is no annotation and no preference model — the label is whether the plan
worked.

---

### Run order

Runs top to bottom with no interaction. Settings > Accelerator > **GPU T4 x2** (or P100).
Expect ~25-40 min end to end on a T4.

### Optional: your own verified questions

The notebook runs fine on synthetic data alone. To also train on questions
really asked of the running copilot, generate them with `make exemplars` and
upload `data/exemplars.jsonl` as a Kaggle dataset named
**`margin-engine-exemplars`**. It is picked up automatically.

---

| Step | Cell |
|---|---|
| Environment + install | 1-2 |
| Compact plan DSL | 3-4 |
| Training corpus | 5-6 |
| Model | 7-8 |
| Train | 9 |
| Evaluate | 10-11 |
| Latency | 12 |
| Export | 13 |


## 1 · Environment


In [ ]:
import os, sys, json, random, re, time, subprocess, textwrap
from pathlib import Path

random.seed(11)
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./out')
OUT.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode: print(r.stdout[-2500:]); print(r.stderr[-2500:])
    return r.returncode

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    print(f'torch {torch.__version__}  cuda={HAS_GPU}')
    if HAS_GPU:
        p = torch.cuda.get_device_properties(0)
        print(f'{p.name}  {p.total_memory/1e9:.1f} GB')
except ImportError:
    HAS_GPU = False
    print('torch not present yet')


## 2 · Install

Unsloth gives roughly 1.5x faster training at ~60% of the VRAM, which is what makes a
4-bit LoRA fit comfortably on a free T4.


In [ ]:
if HAS_GPU:
    sh('pip install -q --no-warn-conflicts unsloth')
    sh('pip install -q --no-deps --upgrade --no-warn-conflicts trl peft accelerate bitsandbytes')
else:
    print('No GPU detected. The data-generation and evaluation cells still run;')
    print('training and export will be skipped.')


## 3 · The plan notation

**This changed, and the reason is the whole point of the rebuild.**

The first version of this notebook emitted a *positional* notation —
`op|cohorts|metrics|dimensions|bin|extra` — chosen because it is short, and
decode latency is proportional to output tokens.

It scored **14.2% exact match** against the grammar tier's 98.4%. The failure
was not comprehension. Nearly every error was *right content, wrong slot*:

```
want  counterfactual|-|-|-|-|rotational_speed_rpm-8
got   counterfactual|-|-|-|rotational_speed_rpm-8|-
```

A positional format makes the model **count pipe separators**, which is close
to the worst thing to ask of a transformer — the same weakness behind every
miscounted letter in a word. It understood the question and put the answer one
field to the left.

So the target is now **JSON**, and two things follow:

1. **Nothing is counted.** Every field is named, so there is no position to
   lose track of.
2. **The shape is guaranteed, not hoped for.** At inference the output is
   constrained by a JSON schema generated from the semantic layer, so a token
   spelling a nonexistent column is *masked before it can be chosen*. Plan
   validity stops being a measured rate and becomes a property of the
   construction.

It costs more tokens per plan. That is the correct trade: a short plan that is
wrong is worth nothing, and latency is a hardware problem while correctness is
not.

**`refuse` is in the vocabulary.** Constraining the output space removes the
model's ability to decline — if every reachable token spells a valid plan then
"I cannot answer that" is unreachable, and the model emits the nearest valid
plan instead. Measured: asked for bearing temperature, a sensor this process
does not have, the constrained model confidently described ambient air. A
constrained decoder needs an explicit escape hatch or it will always answer.


In [ ]:
# ── Semantic layer (mirrors copilot/knowledge/semantic_layer.yaml) ──────────
METRICS = ['air_temp_k','process_temp_k','temp_delta_k','rotational_speed_rpm',
           'torque_nm','tool_wear_min','power_w','overstrain_min_nm','failure']
DIMENSIONS = ['udi','product_type','machine_id','shift']
OPS = ['describe','rate','compare','trend','drivers','root_cause','counterfactual',
       'envelope','forecast','records','data_quality','sql_explore']


def plan_to_target(p):
    """Full plan -> the JSON string the model is trained to emit.

    Canonical: keys in a fixed order, no spaces, empty fields omitted. Two
    plans that mean the same thing must serialise identically or exact-match
    scoring measures formatting rather than understanding.
    """
    out = {'op': p['op']}
    for key in ('metrics', 'group_by', 'filters', 'cohorts'):
        if p.get(key):
            out[key] = p[key]
    if p.get('bin'):
        out['bin'] = p['bin']
    if p.get('effect_size'):
        out['effect_size'] = p['effect_size']
    if p.get('time_grain'):
        out['time_grain'] = p['time_grain']
    # Operator-specific keys, whether they arrive at the top level or nested
    # under params. The first version only looked in params and silently
    # DROPPED a counterfactual's `changes`, so the round-trip check failed and
    # every counterfactual in the corpus would have trained the model to emit a
    # change of nothing.
    for key in ('changes', 'order', 'premise', 'refuse_reason'):
        if p.get(key) is not None:
            out[key] = p[key]
    for k, v in (p.get('params') or {}).items():
        if k not in out and k != 'sql':
            out[k] = v
    return json.dumps(out, separators=(',', ':'), sort_keys=False)


def target_to_plan(s):
    """Back to a plan dict. Trivial now, which is the point — the old notation
    needed a 40-line parser and every line of it was a chance to disagree with
    the generator."""
    return json.loads(s)


# round-trip check
_probe = {'op':'compare',
          'cohorts':[{'name':'failed','filters':[{'field':'failure','op':'=','value':1}]},
                     {'name':'healthy','filters':[{'field':'failure','op':'=','value':0}]}],
          'metrics':['torque_nm','tool_wear_min'],'effect_size':'cohens_d'}
_t = plan_to_target(_probe)
print('target    :', _t)
print('round-trip:', target_to_plan(_t) == _probe)


## 4 · Training corpus

Generated from the semantic layer by templating, then diversified by paraphrase.

**Template overfitting is the main risk of synthetic data**, so every intent carries many
surface forms, entities vary, and word order and politeness are perturbed. Real question
logs (`data/exemplars.jsonl` from the running copilot) are merged in when present, and they
are the higher-quality half — they are questions people actually asked whose plans verified.


In [ ]:
SYNONYM = {
  'torque_nm': ['torque','applied torque','load','torque setting'],
  'rotational_speed_rpm': ['speed','rotational speed','rpm','spindle speed','rotation speed'],
  'tool_wear_min': ['tool wear','wear','tool age','tool life'],
  'temp_delta_k': ['temperature differential','delta t','thermal gradient','temperature difference'],
  'power_w': ['power','mechanical power','power draw','wattage'],
  'overstrain_min_nm': ['overstrain','strain','strain product','accumulated strain'],
  'air_temp_k': ['air temperature','ambient temperature','ambient temp'],
  'process_temp_k': ['process temperature','process temp'],
}
VARIANT = {'L':['L','low quality','low-grade'],'M':['M','medium quality'],'H':['H','high quality']}
POLITE = ['','please ','can you ','could you ','i need to know ','tell me ','show me ']
TAIL   = ['','?',' please','.', ' for me?']

# Quantities a plant plausibly measures that THIS process does not expose.
# Without these the model never learns that declining is an option, and a
# constrained decoder that cannot decline will always answer — with the
# nearest valid plan, about the wrong sensor.
ABSENT = ['bearing temperature','oil pressure','vibration','coolant flow',
          'spindle runout','acoustic level','humidity','motor current',
          'lubricant viscosity','chip load','surface finish','tool cost']

def syn(metric): return random.choice(SYNONYM.get(metric,[metric]))
def wrap(q):
    q = random.choice(POLITE) + q
    return (q[0].upper() + q[1:] if random.random() < .5 else q) + random.choice(TAIL)

FAILED_HEALTHY = [{'name':'failed','filters':[{'field':'failure','op':'=','value':1}]},
                  {'name':'healthy','filters':[{'field':'failure','op':'=','value':0}]}]

def gen_pairs(n_per_intent=90):
    rows = []
    def add(q, plan): rows.append({'question': wrap(q), 'plan': plan})

    for _ in range(n_per_intent):
        m = random.choice(list(SYNONYM))
        add(random.choice([
              f'what are typical {syn(m)} values',
              f'describe the {syn(m)}',
              f'what is the average {syn(m)}',
              f'summarise {syn(m)} across the fleet',
              f'what does {syn(m)} normally look like']),
            {'op':'describe','metrics':[m]})

        v = random.choice(list(VARIANT))
        add(random.choice([
              f'what are operating conditions for {random.choice(VARIANT[v])} variants',
              f'describe conditions on {random.choice(VARIANT[v])} product']),
            {'op':'describe',
             'metrics':['torque_nm','rotational_speed_rpm','tool_wear_min'],
             'filters':[{'field':'product_type','op':'=','value':v}]})

        add(random.choice([
              'what is the overall failure rate','how often do machines fail',
              'what proportion of cycles fail','how many failures are there',
              'what is the breakdown rate']),
            {'op':'rate'})

        d = random.choice(['product_type','shift','machine_id'])
        add(random.choice([
              f'failure rate by {d}', f'break failures down by {d}',
              f'how does the failure rate differ by {d}', f'failures grouped by {d}']),
            {'op':'rate','group_by':[d]})

        m = random.choice(['rotational_speed_rpm','torque_nm','tool_wear_min','power_w'])
        add(random.choice([
              f'why are we seeing more failures at high {syn(m)}',
              f'do failures increase with {syn(m)}',
              f'is {syn(m)} driving more breakdowns',
              f'failure rate across {syn(m)} bands']),
            {'op':'rate','bin':{'field':m,'method':'quantile','bins':5}})

        ms = random.sample(['torque_nm','rotational_speed_rpm','tool_wear_min',
                            'temp_delta_k','power_w'], random.randint(2,4))
        add(random.choice([
              'compare operating conditions of machines that failed versus those that did not',
              'contrast failed and healthy cycles','how do failures differ from normal runs',
              'difference between broken and working machines',
              'set failed against healthy conditions']),
            {'op':'compare','cohorts':FAILED_HEALTHY,'metrics':ms,
             'effect_size':'cohens_d'})

        m = random.choice(['tool_wear_min','rotational_speed_rpm','torque_nm'])
        add(random.choice([
              f'how does failure rate vary with {syn(m)}',
              f'trend of failures against {syn(m)}',
              f'relationship between failures and {syn(m)}',
              f'does failure rate change as {syn(m)} increases']),
            {'op':'trend','bin':{'field':m,'method':'quantile','bins':5}})

        add(random.choice([
              'what drives failures','which variables separate failures from healthy operation',
              'what distinguishes broken machines','biggest factors behind breakdowns',
              'which parameters predict failure']),
            {'op':'drivers'})

        u = random.randint(1, 10000)
        add(random.choice([
              f'why did cycle {u} fail', f'what caused cycle {u} to fail',
              f'root cause for cycle {u}', f'diagnose cycle {u}',
              f'what went wrong on record {u}']),
            {'op':'root_cause','filters':[{'field':'udi','op':'=','value':u}]})

        add(random.choice([
              'what causes failures','what are the main failure modes',
              'attribute the failures','which modes are firing']),
            {'op':'root_cause'})

        delta = random.choice([-10,-8,-5,-3,3,5])
        m = random.choice(['torque_nm','rotational_speed_rpm'])
        verb = 'reduce' if delta < 0 else 'increase'
        unit = 'Nm' if m=='torque_nm' else 'rpm'
        add(random.choice([
              f'what if we {verb} {syn(m)} by {abs(delta)} {unit}',
              f'suppose we {verb} {syn(m)} {abs(delta)} {unit}',
              f'impact of {verb[:-1]}ing {syn(m)} by {abs(delta)} {unit}']),
            {'op':'counterfactual','changes':{m: float(delta)}})

        w, t = random.randint(80,240), random.randint(30,70)
        point = [{'field':'tool_wear_min','op':'=','value':w},
                 {'field':'torque_nm','op':'=','value':t}]
        add(random.choice([
              f'what is the safe torque range at {w} minutes of wear',
              f'operating window at {t} Nm and {w} min wear',
              f'what should i set torque to with {w} min of wear']),
            {'op':'envelope','filters':point})

        add(random.choice([
              f'when will the tool cross the overstrain limit at {w} min wear',
              f'give me the time to crossing at {w} minutes of wear',
              f'how long until failure at {t} Nm',
              f'estimate the crossing time at {w} minutes of wear',
              f'predict when we cross with {w} min of wear']),
            {'op':'forecast','filters':point})

        add(random.choice([
              'show me the cycles closest to failing','list the riskiest records',
              'which cycles are nearest the boundary','give me examples of near misses']),
            {'op':'records','order':'closest_to_failure'})

        add(random.choice([
              'can i trust this data','are there problems with the dataset',
              'data quality report','is the labelling reliable',
              'are the thresholds still accurate']),
            {'op':'data_quality'})

        # Refusals. One in sixteen of the corpus, matching the share of real
        # traffic that asks for something this process does not measure.
        absent = random.choice(ABSENT)
        add(random.choice([
              f'what is the {absent}', f'show me the {absent}',
              f'describe the {absent}', f'what is the average {absent}']),
            {'op':'refuse','refuse_reason':'no such measurement'})
    return rows

pairs = gen_pairs()

# Real logged questions from the running copilot, if uploaded as a dataset.
for candidate in ['/kaggle/input/margin-engine-exemplars/exemplars.jsonl',
                  './data/exemplars.jsonl']:
    path = Path(candidate)
    if path.exists():
        real = 0
        for line in path.read_text().splitlines():
            if not line.strip(): continue
            rec = json.loads(line)
            try:
                pairs.append({'question': rec['question'], 'plan': rec['shape']})
                real += 1
            except Exception:
                pass
        print(f'merged {real} real verified questions from {path}')
        break

# Serialise, deduplicate, then hold out a test split.
for r in pairs:
    r['target'] = plan_to_target(r['plan'])
seen, uniq = set(), []
for r in pairs:
    k = r['question'].lower().strip()
    if k not in seen:
        seen.add(k); uniq.append(r)
random.shuffle(uniq)
split = int(len(uniq)*0.9)
train_rows, test_rows = uniq[:split], uniq[split:]
print(f'{len(uniq)} unique pairs  ->  train {len(train_rows)}  test {len(test_rows)}')
for r in train_rows[:5]: print(f"  {r['question'][:52]:<52} -> {r['target']}")


In [ ]:
# Sanity: the target must be valid JSON, and serialisation must be IDEMPOTENT.
#
# Not `target_to_plan(t) == plan`. Real exemplars merged from the running
# copilot carry model defaults — limit 50, confidence 0.95, empty lists, None
# fields — which plan_to_target correctly strips. Comparing against the raw
# plan flagged 31 of them and would have pushed me to stop stripping, teaching
# the model to emit defaults it never needs to say.
#
# The property that actually matters: the canonical form of a plan is stable.
# Serialise, parse, serialise again, and get the same string.
bad = []
for r in uniq:
    try:
        again = plan_to_target(target_to_plan(r['target']))
        if again != r['target']:
            bad.append((r['target'], again))
    except Exception as e:
        bad.append((r['target'], str(e)))
print(f'non-canonical targets: {len(bad)}')
for b in bad[:5]: print(' ', b)
assert not bad, 'fix the generator before training'

refusals = sum(1 for r in uniq if r['plan'].get('op') == 'refuse')
print(f'refusal examples: {refusals} of {len(uniq)} ({refusals/len(uniq):.0%})')
import statistics
print(f"median target length: {statistics.median(len(r['target']) for r in uniq):.0f} chars")


## 5 · Model

A closed output space of 12 ops and ~13 names does not need a large model — it needs a
reliable one. Candidates are tried in order and the first that loads is used, so the
notebook completes even if a model id has moved.

Apache-2.0 licensed throughout, which matters for on-premise industrial deployment.


In [ ]:
MAX_SEQ = 512
CANDIDATES = [
    'unsloth/Qwen3-1.7B-unsloth-bnb-4bit',
    'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
    'unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    'unsloth/Llama-3.2-1B-Instruct-bnb-4bit',
    'unsloth/gemma-2-2b-it-bnb-4bit',
]
model = tokenizer = MODEL_ID = None

if HAS_GPU:
    from unsloth import FastLanguageModel
    for mid in CANDIDATES:
        try:
            print(f'loading {mid} ...')
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=mid, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None)
            MODEL_ID = mid
            print(f'  loaded {mid}')
            break
        except Exception as e:
            print(f'  unavailable ({type(e).__name__}: {str(e)[:110]})')
    assert model is not None, 'no candidate model could be loaded'
else:
    print('skipped (no GPU)')


In [ ]:
if HAS_GPU:
    model = FastLanguageModel.get_peft_model(
        model,
        r=16, lora_alpha=32, lora_dropout=0.0, bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj',
                        'gate_proj','up_proj','down_proj'],
        use_gradient_checkpointing='unsloth', random_state=11)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'trainable {trainable/1e6:.1f}M of {total/1e6:.0f}M  ({trainable/total:.2%})')


## 6 · Prompt format

The system prompt carries the vocabulary, so the model learns the *mapping pattern* rather
than memorising the schema. Adding a metric later then needs no retraining — the new name
simply appears in the prompt.


In [ ]:
SYSTEM = (
 'Translate the engineer question into one Analysis Plan as JSON.\n'
 'Output the JSON object only.\n'
 f"ops: {' '.join(OPS)} refuse\n"
 f"metrics: {' '.join(METRICS)}\n"
 f"dimensions: {' '.join(DIMENSIONS)}\n"
 'keys: op, metrics, group_by, filters, cohorts, bin, effect_size, changes, order\n'
 'filters: [{"field":..,"op":"=","value":..}]   bin: {"field":..,"method":"quantile","bins":5}\n'
 'If the question names a quantity not listed above, emit '
 '{"op":"refuse","refuse_reason":"no such measurement"}.'
)

def to_text(row, tok):
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':row['question']},
            {'role':'assistant','content':row['target']}]
    return tok.apply_chat_template(msgs, tokenize=False)

def to_prompt(question, tok):
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':question}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

if HAS_GPU:
    from datasets import Dataset
    train_ds = Dataset.from_list([{'text': to_text(r, tokenizer)} for r in train_rows])
    lens = [len(tokenizer(t['text'])['input_ids']) for t in train_ds.select(range(min(200,len(train_ds))))]
    print(f'{len(train_ds)} examples   median {sorted(lens)[len(lens)//2]} tokens   max {max(lens)}')
    print(train_ds[0]['text'][:600])


## 7 · Train


In [ ]:
if HAS_GPU:
    from trl import SFTTrainer, SFTConfig
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_ds,
        args=SFTConfig(
            dataset_text_field='text', max_seq_length=MAX_SEQ,
            per_device_train_batch_size=4, gradient_accumulation_steps=4,
            warmup_steps=10, num_train_epochs=2, learning_rate=2e-4,
            logging_steps=20, optim='adamw_8bit', weight_decay=0.01,
            lr_scheduler_type='linear', seed=11, report_to='none',
            output_dir=str(OUT/'checkpoints'), save_strategy='no'))
    t0 = time.time()
    stats = trainer.train()
    print(f'\ntrained in {(time.time()-t0)/60:.1f} min   final loss {stats.training_loss:.4f}')


## 8 · Evaluate

Two metrics, held out:

- **DSL exact match** — the strict one. Did it emit precisely the right plan line?
- **Plan validity** — did the output parse into a structurally valid plan at all?

Op accuracy is reported separately, because choosing the right *analysis* is most of the
job; a missing metric is recoverable, a wrong op is not.


In [ ]:
def generate(question, max_new=192):
    inputs = tokenizer([to_prompt(question, tokenizer)], return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip()

if HAS_GPU:
    FastLanguageModel.for_inference(model)
    exact = valid = op_ok = 0
    refusals_right = refusals_total = 0
    misses = []
    for r in test_rows:
        pred = generate(r['question'])
        if pred == r['target']: exact += 1
        else: misses.append((r['question'], r['target'], pred))
        try:
            plan = target_to_plan(pred); valid += 1
            if plan.get('op') == r['plan']['op']: op_ok += 1
        except Exception:
            plan = None
        if r['plan']['op'] == 'refuse':
            refusals_total += 1
            if plan and plan.get('op') == 'refuse': refusals_right += 1
    n = len(test_rows)
    print(f'held-out n            {n}')
    print(f'exact match           {exact/n:.3f}')
    print(f'plan validity         {valid/n:.3f}   <- 1.000 under constrained decoding')
    print(f'op accuracy           {op_ok/n:.3f}')
    if refusals_total:
        print(f'refusal recall        {refusals_right/refusals_total:.3f} '
              f'({refusals_right}/{refusals_total})')
    print()
    print('NOTE. Validity here is measured on UNCONSTRAINED generation, which is')
    print('the honest number for the fine-tune itself. At inference the planner')
    print('applies a JSON schema generated from the semantic layer, so validity')
    print('is 1.000 by construction and this figure measures how much the model')
    print('needed that help. A large gap means the training did not take.')
    print()
    for q,g,pr in misses[:8]:
        print(f'  Q    {q[:66]}'); print(f'  want {g}'); print(f'  got  {pr}\n')


## 9 · Latency

This is the number that decides whether the adapter is worth deploying. Compare it against
the tiers it would sit behind:

| Tier | Latency |
|---|---|
| plan cache | ~0 ms |
| grammar | ~1 ms |
| verified exemplars | ~1 ms |
| **this adapter** | **measured below** |
| frontier API | ~400-600 ms |

The adapter is not competing with the cheap tiers — it replaces the **API call** on
genuinely novel questions, and removes the network from the path entirely.


In [ ]:
if HAS_GPU:
    probes = [r['question'] for r in test_rows[:12]] or ['what is the overall failure rate']
    generate(probes[0])                       # warm
    times, toks = [], []
    for q in probes:
        t0 = time.time(); out = generate(q); times.append(time.time()-t0)
        toks.append(len(tokenizer(out)['input_ids']))
    times.sort()
    p50, p95 = times[len(times)//2], times[min(int(len(times)*.95), len(times)-1)]
    print(f'output tokens  median {sorted(toks)[len(toks)//2]}')
    print(f'latency p50    {p50*1000:.0f} ms')
    print(f'latency p95    {p95*1000:.0f} ms')
    print(f'decode rate    {sum(toks)/sum(times):.0f} tok/s')
    print(f'\nvs a ~450 ms frontier call: {450/(p50*1000):.1f}x faster, and no network')


## 10 · Export

Three artifacts:

1. **LoRA adapter** — a few tens of MB, loads on top of the base model
2. **Merged 16-bit** — for vLLM / TGI serving, where XGrammar can constrain decoding
3. **GGUF q4_k_m** — for `llama.cpp` / Ollama, i.e. running it on a laptop or an edge box

For the copilot, point `COPILOT_PROVIDER=ollama` and `COPILOT_SLM_MODEL` at the GGUF.


In [ ]:
if HAS_GPU:
    adapter = OUT/'margin-planner-lora'
    model.save_pretrained(str(adapter)); tokenizer.save_pretrained(str(adapter))
    print(f'adapter -> {adapter}')
    try:
        model.save_pretrained_gguf(str(OUT/'margin-planner-gguf'), tokenizer,
                                   quantization_method='q4_k_m')
        print('gguf    -> ok')
    except Exception as e:
        print(f'gguf skipped ({type(e).__name__}: {str(e)[:120]})')

    (OUT/'planner_card.json').write_text(json.dumps({
        'base_model': MODEL_ID,
        'task': 'question -> Analysis Plan (compact DSL)',
        'train_examples': len(train_rows),
        'held_out': len(test_rows),
        'system_prompt': SYSTEM,
        'dsl_format': 'op|cohorts|metrics|dimensions|bin|extra',
        'note': 'Plans are validated against the semantic layer before execution and every '
                'number in the final answer is computed and verified downstream. This model '
                'never emits a number.',
    }, indent=2))
    print('card    -> planner_card.json')
    print('\nfiles:'); [print(' ', p.name) for p in sorted(OUT.iterdir())]


## 11 · Wiring it back in

```bash
# local serving
ollama create margin-planner -f Modelfile      # FROM ./margin-planner-gguf/...Q4_K_M.gguf
export COPILOT_PROVIDER=ollama
export COPILOT_SLM_MODEL=margin-planner
make chat
```

**Promotion is gated, not automatic.** The adapter ships only if it beats the incumbent on
the golden set:

```bash
make eval          # hard gates must stay green:
                   #   unsourced_numeral_rate  0.000
                   #   numeric_exactness       1.000
                   #   refusal_correctness     1.000
```

If it regresses, roll back to the verified-exemplar store — which never stopped working,
because it needs no training at all.
